# Two Agents: Code Writer & Executor — Local (Ollama, no API key)

## Pre-requisite (do this ONCE outside Jupyter)
1. Download Ollama: https://ollama.com/download
2. Install and run it — it starts a local server on port 11434
3. Open a terminal and run: `ollama pull llama3.2`
4. Come back here and run cells top to bottom

In [1]:
%pip install -q autogen-agentchat autogen-ext[openai] openai

Note: you may need to restart the kernel to use updated packages.


In [2]:
import asyncio
from pathlib import Path
from autogen_agentchat.agents import AssistantAgent, CodeExecutorAgent
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor

import autogen_agentchat
print('autogen-agentchat version:', autogen_agentchat.__version__)

autogen-agentchat version: 0.7.5


In [3]:
# Ollama exposes an OpenAI-compatible API on localhost — no key needed
ollama_client = OpenAIChatCompletionClient(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
    api_key="ollama",          # required by the lib but ignored by Ollama
    model_capabilities={
        "vision": False,
        "function_calling": False,
        "json_output": False,
    },
)
print('Ollama client ready')

Ollama client ready


C:\Users\lhari\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [4]:
# Quick connectivity test — confirms Ollama is running
import urllib.request, json as _json
try:
    with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
        tags = _json.loads(r.read())
        models = [m['name'] for m in tags.get('models', [])]
        print('Ollama is running. Available models:', models)
        if not any('llama3.2' in m for m in models):
            print('\n⚠️  llama3.2 not found. Run in terminal: ollama pull llama3.2')
except Exception as e:
    print('❌ Ollama not reachable:', e)
    print('Make sure Ollama is installed and running (ollama serve)')

Ollama is running. Available models: ['llama3.2:latest']


## Create Agents

### Agent 1 — Code Executor

In [5]:
work_dir = Path("coding")
work_dir.mkdir(exist_ok=True)

executor = LocalCommandLineCodeExecutor(work_dir=work_dir)

code_executor_agent = CodeExecutorAgent(
    name="code_executor_agent",
    code_executor=executor,
)
print('Code executor agent ready')

Code executor agent ready


C:\Users\lhari\AppData\Local\Temp\ipykernel_21772\416409990.py:4: UserWarning: Using LocalCommandLineCodeExecutor may execute code on the local machine which can be unsafe. For security, it is recommended to use DockerCommandLineCodeExecutor instead. To install Docker, visit: https://docs.docker.com/get-docker/
  executor = LocalCommandLineCodeExecutor(work_dir=work_dir)
C:\Users\lhari\AppData\Local\Temp\ipykernel_21772\416409990.py:4: UserWarning: The current event loop policy is not WindowsProactorEventLoopPolicy. This may cause issues with subprocesses. Try setting the event loop policy to WindowsProactorEventLoopPolicy. For example: `asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())`. See https://docs.python.org/3/library/asyncio-eventloop.html#asyncio.ProactorEventLoop.
  executor = LocalCommandLineCodeExecutor(work_dir=work_dir)
C:\Users\lhari\AppData\Local\Temp\ipykernel_21772\416409990.py:6: UserWarning: No approval function set for CodeExecutorAgent. This

### Agent 2 — Code Writer

In [6]:
code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
Always suggest Python code in a python coding block.
Rules:
- Only one code block per response.
- Use print() for all outputs.
- Do not ask users to modify code.
- Do not use incomplete code.
- Check the execution result returned.
- When the task is fully complete, reply with TERMINATE.
"""

code_writer_agent = AssistantAgent(
    name="code_writer",
    model_client=ollama_client,
    system_message=code_writer_system_message,
)
print('Code writer agent ready')

Code writer agent ready


## Run the Two-Agent Chat

In [7]:
termination = TextMentionTermination("TERMINATE") | MaxMessageTermination(10)

team = RoundRobinGroupChat(
    [code_writer_agent, code_executor_agent],
    termination_condition=termination,
)

task = (
    "Write Python code for permutation for the word ALGEBRA. "
    "Use an optimised way to calculate it. I want the final result count."
)

result = await Console(team.run_stream(task=task))

---------- TextMessage (user) ----------
Write Python code for permutation for the word ALGEBRA. Use an optimised way to calculate it. I want the final result count.
---------- TextMessage (code_writer) ----------
```python
def permute_word(word):
    import math
    # Calculate the factorial of a number using math.comb function (Python 3.8+)
    if hasattr(math, 'comb'):  
        n = len(word)
        return math.comb(n, n)

# Define the word
word = "ALGEBRA"

# Check if the math.comb function is available
if hasattr(math, 'comb'):
    print("The total number of permutations is:", permute_word(word))
else:
    # Calculate the factorial for earlier versions of Python (3.8+)
    import math, functools as ft
    
    def comb(x, y):
        return math.factorial(x) // (math.factorial(y) * math.factorial(x - y))

    n = len(word)
    result = 0
    for i in range(n):
        # Multiply the previous total by n-previous-i+1/i!
        result += ft.reduce(lambda x, y: x*y, comb(n -i -1, i)

## Inspect Results

In [8]:
for msg in result.messages:
    print(f"[{msg.source}]: {str(msg.content)[:200]}")
    print()

[user]: Write Python code for permutation for the word ALGEBRA. Use an optimised way to calculate it. I want the final result count.

[code_writer]: ```python
def permute_word(word):
    import math
    # Calculate the factorial of a number using math.comb function (Python 3.8+)
    if hasattr(math, 'comb'):  
        n = len(word)
        return 



In [9]:
print('=== FINAL RESULT ===')
print(result.messages[-1].content)

=== FINAL RESULT ===
```python
def permute_word(word):
    import math
    # Calculate the factorial of a number using math.comb function (Python 3.8+)
    if hasattr(math, 'comb'):  
        n = len(word)
        return math.comb(n, n)

# Define the word
word = "ALGEBRA"

# Check if the math.comb function is available
if hasattr(math, 'comb'):
    print("The total number of permutations is:", permute_word(word))
else:
    # Calculate the factorial for earlier versions of Python (3.8+)
    import math, functools as ft
    
    def comb(x, y):
        return math.factorial(x) // (math.factorial(y) * math.factorial(x - y))

    n = len(word)
    result = 0
    for i in range(n):
        # Multiply the previous total by n-previous-i+1/i!
        result += ft.reduce(lambda x, y: x*y, comb(n -i -1, i))
    
    print("The total number of permutations is:", result)

TERMINATE
```


In [10]:
print('Stop reason:', result.stop_reason)

Stop reason: Text 'TERMINATE' mentioned
